# What If? — Counterfactual World Generator
Give a model a real-world rule, ask for a world where it doesn't exist, and compare **generic** vs **structured multi-step** prompting.

**Runtime → Change runtime type → T4 GPU** before running (FLAN-T5-large also works on CPU, just slower).

## 1. Setup

In [ ]:
# Option A: clone your GitHub repo (set REPO_URL). Option B: leave it empty and upload the project zip.
REPO_URL = "https://github.com/shre-yeah/What_If_Counterfactual_World_Simulator.git"

import os, sys
if REPO_URL:
    !rm -rf project && git clone -q {REPO_URL} project
else:
    from google.colab import files
    up = files.upload()  # upload project.zip (zip of this folder)
    !unzip -q -o project.zip -d project
root = next(p for p in ("project", "project/Counterfactual World Simulator") if os.path.exists(f"{p}/src"))
sys.path.insert(0, f"{root}/src")
!pip install -q -r "{root}/requirements.txt"

## 2. Load the model

In [ ]:
from cfworld.model import Generator
gen = Generator("google/flan-t5-large")   # swap for "google/flan-t5-base" if memory is tight

## 3. Try one world, both ways

In [ ]:
from cfworld import generic_world, structured_world

rule = "humans never needed sleep"
g = generic_world(gen, rule, seed=0)
s = structured_world(gen, rule, seed=0)

print("GENERIC")
for c in g.consequences: print(" -", c.text)
print("\nSTRUCTURED  (premise:", s.premise, ")")
for c in s.consequences: print(f" - [{c.domain}] {c.text}")

## 4. Run the experiment
For each rule and method: **K=5 runs** (different seeds).
- **consistency** — do repeated runs on the same rule agree? (higher = more repeatable)
- **semantic_diversity** — how different are consequences within one world? (higher = more varied)
- **distinct_2** — lexical variety (unique bigrams / total)
- **domain_coverage / domain_entropy** — how evenly the five life domains are covered

Both methods produce the same number of consequences per world, so differences come from the prompting strategy.

In [ ]:
from cfworld import run_experiment
df, worlds = run_experiment(gen, k=5, out_dir="results")
df

In [ ]:
summary = df.groupby("method").mean(numeric_only=True).round(3)
summary

In [ ]:
import matplotlib.pyplot as plt
summary.T.plot.bar(figsize=(9, 4), rot=30, title="Generic vs structured prompting")
plt.tight_layout(); plt.show()

## 5. Read the outputs — numbers can hide bad text

In [ ]:
for method in ("generic", "structured"):
    print("=====", method)
    for w in worlds[("nobody could lie", method)][:2]:
        for c in w.consequences: print(" -", c.text)
        print()

## 6. Next ideas
- `structured_world(..., ripple=True)` adds second-order consequences.
- Try `flan-t5-base` vs `-large` vs `-xl`, and vary temperature.
- Add your own rules to `cfworld/prompts.py::RULES`.
- Download `results/` (summary.csv, worlds.json) from the Files sidebar.